In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

s:\NeuroMate\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_path = "../Data/sample_report_2.pdf"
loader = PyPDFLoader(pdf_path)
pages = loader.load()
print(f"Number of pages: {len(pages)}")
print(f"Content of the first page:\n{pages[0].page_content}")

Number of pages: 5
Content of the first page:
MedGenome Labs Ltd. 
3rd Floor, Narayana Nethralaya Building, Narayana Health City, 
#258/A, Bommasandra, Hosur Road, Bangalore - 560 099, India. 
Tel : +91 (0)80 67154989 / 990, Web: www.medgenome.com 
 
 
Page 1 of 5 
Name/Sample ID: Shivanshu Gupta/7739815 
 
DNA TEST REPORT - MEDGENOME LABS 
Full Name / Ref No: SHIVANSHU GUPTA Order ID/Sample ID: 527250/7739815 
Gender: Male Sample Type: Blood 
Date of Birth / Age: 19 years Date of Sample Collection: 3rd November 2022 
Referring Clinician: Dr. Vishnu V. Y.,  
AIIMS,  
New Delhi 
Date of Sample Receipt: 4th November 2022 
Date of Order Booking: 5th November 2022 
Date of Report: 2nd December 2022 
Test Requested: Whole Exome Sequencing 
CLINICAL DIAGNOSIS / SYMPTOMS / HISTORY 
Mr. Shivanshu Gupta, presented with clinical indications of delayed motor milestones, progressive gait difficulties, bilateral 
proximal upper and lower limb weakness, areflexia, scoliosis, decreased sensation in b

In [8]:
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate

# Initialize local model
llm = Ollama(model="llama3.2:3b")

# Define your extraction logic
template = '''
You are a helpful assistant that extracts key information from medical reports.
You are given the following genetic report:
{text}

Your task is to extract the following information:
1. Patient Name
2. Age(in years, null if not found)
3. Gender(Male/Female, null if not found)
4. Gene Name
5. Variant
6. Variant Type
7. Zygosity
8. Classification
9. Disease Name (if mentioned else null)

The output must be in JSON format with the keys as mentioned above. If any information is not found, the value should be null.
No extra text should be included in the output. Only the JSON object should be returned.

'''
prompt = PromptTemplate.from_template(template)

chain = prompt | llm
result = chain.invoke({"text": pages[0].page_content})

In [9]:
print(result)

{"PatientName": "Anonymous", "Age": null, "Gender": "Male", "GeneName": "PMP22", "Variant": "17p12 duplication (1.4 Mb region including entire PMP22 gene)", "VariantType": "duplication", "Zygosity": "heterozygous", "Classification": "Pathogenic", "DiseaseName": "CMT1A"}


In [10]:
test_report2 = PyPDFLoader("../Data/sample_report_2.pdf")
pages2 = test_report2.load()
print(f"Number of pages: {len(pages2)}")
print(f"Content of the first page:\n{pages2[0].page_content}")

Number of pages: 5
Content of the first page:
MedGenome Labs Ltd. 
3rd Floor, Narayana Nethralaya Building, Narayana Health City, 
#258/A, Bommasandra, Hosur Road, Bangalore - 560 099, India. 
Tel : +91 (0)80 67154989 / 990, Web: www.medgenome.com 
 
 
Page 1 of 5 
Name/Sample ID: Shivanshu Gupta/7739815 
 
DNA TEST REPORT - MEDGENOME LABS 
Full Name / Ref No: SHIVANSHU GUPTA Order ID/Sample ID: 527250/7739815 
Gender: Male Sample Type: Blood 
Date of Birth / Age: 19 years Date of Sample Collection: 3rd November 2022 
Referring Clinician: Dr. Vishnu V. Y.,  
AIIMS,  
New Delhi 
Date of Sample Receipt: 4th November 2022 
Date of Order Booking: 5th November 2022 
Date of Report: 2nd December 2022 
Test Requested: Whole Exome Sequencing 
CLINICAL DIAGNOSIS / SYMPTOMS / HISTORY 
Mr. Shivanshu Gupta, presented with clinical indications of delayed motor milestones, progressive gait difficulties, bilateral 
proximal upper and lower limb weakness, areflexia, scoliosis, decreased sensation in b

In [11]:
result2 = chain.invoke({"text": pages2[0].page_content})

In [12]:
print(result2)

{"Patient Name": "Shivanshu Gupta", "Age": "19", "Gender": "Male", "Gene Name": "SH3TC2", "Variant": "c.3325C>T", "Variant Type": "SNV", "Zygosity": "Homozygous", "Classification": "Pathogenic", "Disease Name": "Charcot-Marie-Tooth disease type 4C"}


In [7]:
print(pages[0].page_content)

MedGenome Labs Ltd. 
3rd Floor, Narayana Nethralaya Building, Narayana Health City, 
#258/A, Bommasandra, Hosur Road, Bangalore - 560 099, India. 
Tel : +91 (0)80 67154989 / 990, Web: www.medgenome.com 
 
 
Page 1 of 5 
Name/Sample ID: Shivanshu Gupta/7739815 
 
DNA TEST REPORT - MEDGENOME LABS 
Full Name / Ref No: SHIVANSHU GUPTA Order ID/Sample ID: 527250/7739815 
Gender: Male Sample Type: Blood 
Date of Birth / Age: 19 years Date of Sample Collection: 3rd November 2022 
Referring Clinician: Dr. Vishnu V. Y.,  
AIIMS,  
New Delhi 
Date of Sample Receipt: 4th November 2022 
Date of Order Booking: 5th November 2022 
Date of Report: 2nd December 2022 
Test Requested: Whole Exome Sequencing 
CLINICAL DIAGNOSIS / SYMPTOMS / HISTORY 
Mr. Shivanshu Gupta, presented with clinical indications of delayed motor milestones, progressive gait difficulties, bilateral 
proximal upper and lower limb weakness, areflexia, scoliosis, decreased sensation in bilateral feet and pes cavus. Examination 
reve

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
for i, page in enumerate(pages):
    chunks = splitter.split_text(page.page_content)
    print(f"Page {i+1} has {len(chunks)} chunks.")
    print(chunks[0])
    print("-----------------------------------")

Page 1 has 3 chunks.
MedGenome Labs Ltd. 
3rd Floor, Narayana Nethralaya Building, Narayana Health City, 
#258/A, Bommasandra, Hosur Road, Bangalore - 560 099, India. 
Tel : +91 (0)80 67154989 / 990, Web: www.medgenome.com 
 
 
Page 1 of 5 
Name/Sample ID: Shivanshu Gupta/7739815 
 
DNA TEST REPORT - MEDGENOME LABS 
Full Name / Ref No: SHIVANSHU GUPTA Order ID/Sample ID: 527250/7739815 
Gender: Male Sample Type: Blood 
Date of Birth / Age: 19 years Date of Sample Collection: 3rd November 2022 
Referring Clinician: Dr. Vishnu V. Y.,  
AIIMS,  
New Delhi 
Date of Sample Receipt: 4th November 2022 
Date of Order Booking: 5th November 2022 
Date of Report: 2nd December 2022 
Test Requested: Whole Exome Sequencing 
CLINICAL DIAGNOSIS / SYMPTOMS / HISTORY 
Mr. Shivanshu Gupta, presented with clinical indications of delayed motor milestones, progressive gait difficulties, bilateral 
proximal upper and lower limb weakness, areflexia, scoliosis, decreased sensation in bilateral feet and pes cav